In [28]:
import yfinance as yf
from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import anthropic
from dotenv import load_dotenv
import numpy as np
 
df = yf.download("AAPL", period="1y")
# print(df.shape)
# print(df.head())
# print(df.columns)

df['Tomorrow'] = df['Close'].shift(-1)
df.dropna(inplace=True)

X_train, X_test, y_train, y_test = train_test_split(df[['Open', 'High', 'Low', 'Close', 'Volume']], df['Tomorrow'], test_size=0.2, train_size=0.8, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

predictions = model.predict(X_test)
actual = y_test.values


def get_analyst_opinion(current_price, predicted_price, ticker):
    percent_change = ((predicted_price - current_price) / np.abs(current_price)) * 100
    
    client = anthropic.Anthropic()
    
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system="you are a professional financial analyst. you will provide recomendation of the trade tomorrow (buy/sell). Keep the response straight and concise",
        messages=[
            {"role": "user", "content": f"i am analyzing {ticker}. My cuurent price is {current_price} and i have used linear regression to predict the price for tomorrow. The precentage change between the actual and predicted value is {percent_change}"}
        ]
    )

    print(message.content[0].text)

get_analyst_opinion(actual[-1], predictions[-1], "AAPL" )





[*********************100%***********************]  1 of 1 completed


0.7705225516630818
# AAPL Trade Recommendation

**SELL / TAKE PROFITS**

**Rationale:**
- Your linear regression predicts a **1.30% price increase** tomorrow
- This modest upside doesn't justify holding given:
  - Limited risk/reward ratio
  - Potential profit-taking pressure at resistance
  - Safer to lock in gains on a +1.3% prediction

**Alternative:** If you have a longer holding period, a 1.3% gain could be acceptable. But for a single-day trade, the margin is thin.

**Action:** Sell at market open or at resistance levels to secure gains.
